# Gold Layer — Banking Data Model

## Objective
Transform validated Silver data into business-ready Delta tables for analytics and reporting.

## Data Model
- **dim_customer:** Customer dimension maintained using SCD Type 1.
- **dim_account:** Account dimension maintained using SCD Type 1.
- **fact_transactions:** Valid banking transactions at transaction-level granularity.
- **gold_daily_transactions:** Daily transaction counts and amounts.
- **gold_fraud_summary:** Fraud-flag statistics grouped by transaction date and channel.

## Processing Strategy
- Initialize Gold tables from Silver using Delta CTAS.
- Use MERGE to synchronize transaction facts and dimensions.
- Recalculate daily and fraud aggregates from Gold transaction facts.
- Support repeatable processing without inserting duplicate transaction IDs.

## Design Considerations
- Gold fact and dimension MERGEs currently read complete Silver tables.
- Transaction records do not contain account_id, so a direct transaction-to-account relationship is not modeled.
- Summary MERGEs do not currently remove obsolete aggregate groups.

## Validation
Batch 07 incremental testing:
- 3 raw transactions ingested into Bronze.
- 2 valid transactions accepted into Silver.
- 1 invalid transaction quarantined.
- 2 transactions inserted into Gold.
- Daily Gold summary verified: 2 transactions totaling 1,250.00 on 2026-11-11.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS banking_project.gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_project.gold.dim_customer
USING DELTA
AS
SELECT
    customer_id,
    customer_name,
    customer_city
FROM banking_project.silver.silver_customers;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_project.gold.fact_transactions
USING DELTA
AS
SELECT
    transaction_id,
    customer_id,
    transaction_date,
    transaction_time,
    account_type,
    transaction_type,
    transaction_amount,
    transaction_direction,
    account_balance,
    merchant_category,
    transaction_status,
    channel,
    is_fraud,
    transaction_hour
FROM banking_project.silver.silver_transactions;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_project.gold.dim_account
USING DELTA
AS
SELECT
    account_id,
    customer_id,
    account_type
FROM banking_project.silver.silver_accounts;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_project.gold.gold_daily_transactions
USING DELTA
AS
SELECT
    transaction_date,
    COUNT(*) AS total_transactions,
    SUM(transaction_amount) AS total_amount,
    AVG(transaction_amount) AS avg_transaction_amount,
    SUM(
        CASE
            WHEN UPPER(transaction_status) = 'SUCCESS'
            THEN 1 ELSE 0
        END
    ) AS successful_transactions,
    SUM(
        CASE
            WHEN UPPER(transaction_status) = 'FAILED'
            THEN 1 ELSE 0
        END
    ) AS failed_transactions
FROM banking_project.gold.fact_transactions
GROUP BY transaction_date;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_project.gold.gold_fraud_summary
USING DELTA
AS
SELECT
    transaction_date,
    channel,
    COUNT(*) AS total_transactions,
    SUM(
        CASE
            WHEN LOWER(TRIM(is_fraud)) IN ('true', '1', 'yes')
            THEN 1 ELSE 0
        END
    ) AS flagged_fraud_transactions,
    SUM(
        CASE
            WHEN LOWER(TRIM(is_fraud)) IN ('true', '1', 'yes')
            THEN transaction_amount
            ELSE 0
        END
    ) AS flagged_fraud_amount
FROM banking_project.gold.fact_transactions
GROUP BY transaction_date, channel;

In [0]:
%sql
MERGE INTO banking_project.gold.fact_transactions AS target
USING (
    SELECT
        transaction_id,
        customer_id,
        transaction_date,
        transaction_time,
        account_type,
        transaction_type,
        transaction_amount,
        transaction_direction,
        account_balance,
        merchant_category,
        transaction_status,
        channel,
        is_fraud,
        transaction_hour
    FROM banking_project.silver.silver_transactions
) AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO banking_project.gold.dim_customer AS target
USING banking_project.silver.silver_customers AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND NOT (
    target.customer_name <=> source.customer_name
    AND target.customer_city <=> source.customer_city
)
THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.customer_city = source.customer_city

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    customer_city
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.customer_city
);

In [0]:
%sql
MERGE INTO banking_project.gold.dim_account AS target
USING banking_project.silver.silver_accounts AS source
ON target.account_id = source.account_id

WHEN MATCHED AND NOT (
    target.customer_id <=> source.customer_id
    AND target.account_type <=> source.account_type
)
THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.account_type = source.account_type

WHEN NOT MATCHED THEN
INSERT (
    account_id,
    customer_id,
    account_type
)
VALUES (
    source.account_id,
    source.customer_id,
    source.account_type
);

In [0]:
%sql
MERGE INTO banking_project.gold.gold_daily_transactions AS target
USING (
    SELECT
        transaction_date,
        COUNT(*) AS total_transactions,
        SUM(transaction_amount) AS total_amount,
        AVG(transaction_amount) AS avg_transaction_amount,
        SUM(
            CASE WHEN UPPER(transaction_status) = 'SUCCESS'
            THEN 1 ELSE 0 END
        ) AS successful_transactions,
        SUM(
            CASE WHEN UPPER(transaction_status) = 'FAILED'
            THEN 1 ELSE 0 END
        ) AS failed_transactions
    FROM banking_project.gold.fact_transactions
    GROUP BY transaction_date
) AS source

ON target.transaction_date = source.transaction_date

WHEN MATCHED AND NOT (
    target.total_transactions <=> source.total_transactions
    AND target.total_amount <=> source.total_amount
    AND target.avg_transaction_amount <=> source.avg_transaction_amount
    AND target.successful_transactions <=> source.successful_transactions
    AND target.failed_transactions <=> source.failed_transactions
)
THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO banking_project.gold.gold_fraud_summary AS target
USING (
    SELECT
        transaction_date,
        channel,
        COUNT(*) AS total_transactions,
        SUM(
            CASE
                WHEN LOWER(TRIM(is_fraud)) IN ('1', 'true', 'yes')
                THEN 1 ELSE 0
            END
        ) AS flagged_fraud_transactions,
        SUM(
            CASE
                WHEN LOWER(TRIM(is_fraud)) IN ('1', 'true', 'yes')
                THEN transaction_amount
                ELSE 0
            END
        ) AS flagged_fraud_amount
    FROM banking_project.gold.fact_transactions
    GROUP BY transaction_date, channel
) AS source

ON target.transaction_date = source.transaction_date
AND target.channel <=> source.channel

WHEN MATCHED AND NOT (
    target.total_transactions <=> source.total_transactions
    AND target.flagged_fraud_transactions <=> source.flagged_fraud_transactions
    AND target.flagged_fraud_amount <=> source.flagged_fraud_amount
)
THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT transaction_id) AS unique_transactions
FROM banking_project.gold.fact_transactions;

In [0]:
%sql
SELECT
    transaction_date,
    COUNT(*) AS total_transactions,
    SUM(transaction_amount) AS total_amount
FROM banking_project.gold.fact_transactions
GROUP BY transaction_date
ORDER BY transaction_date;

In [0]:
%sql
SELECT *
FROM banking_project.gold.gold_fraud_summary;